# Tutorial 5 — Advanced Plotting

**What you'll learn:** How to customise every ForMoSA plot, extract numerical results
programmatically, compute derived statistics (χ²_red, log-evidence), and save
publication-quality figures.

**No fitting required.** This tutorial loads pre-computed results from Tutorial 2
(AB Pic b, VLT/SINFONI K-band) — a small JSON file committed to the repository.

**Estimated runtime:** < 30 seconds (no nested sampling).


## Section 0: Setup

In [ ]:
import sys
try:
    import ForMoSA
    print(f"ForMoSA {ForMoSA.__version__} — OK")
except ImportError:
    raise ImportError("pip install ForMoSA && conda install dask netCDF4 bottleneck")
print(f"Python {sys.version.split()[0]}")


In [ ]:
# Load pre-computed results — no fitting needed
import json
from pathlib import Path
from ForMoSA.nested_sampling.results import NSResults

RESULTS_FILE = Path(".").resolve() / "ns_results.json"

if not RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Results file not found: {RESULTS_FILE}\n"
        "Make sure you are running this notebook from the plotting/ directory."
    )

with open(RESULTS_FILE) as f:
    data = json.load(f)

results = NSResults.from_dict(data)

print(f"Results loaded: {results.samples.shape[0]} samples")
print(f"Free parameters: {results.free_parameters}")
print(f"Burn-in: {results.burn_in} samples")


## Section 1: Reading results

`NSResults` stores everything from the nested sampling run. The key attributes:

| Attribute | Type | Description |
|-----------|------|-------------|
| `free_parameters` | `list[str]` | Names of fitted parameters |
| `samples` | `(N, n_params) ndarray` | All samples drawn by the sampler |
| `weights` | `(N,) ndarray` | Bayesian weight of each sample |
| `logl` | `(N,) ndarray` | Log-likelihood of each sample |
| `logvol` | `(N,) ndarray` | Log prior volume at each step |
| `logz` | `[float, float]` | Log-evidence and its uncertainty |
| `burn_in` | `int` | Index before which samples are discarded |


In [ ]:
# Numerical summary: median ± 1σ for each parameter
print(results.summary(sigma=1))
print()

# Access individual statistics programmatically
medians = results.median_parameters
intervals = results._interval(sigma=1)

for name in results.free_parameters:
    med = medians[name]
    lo, hi = intervals[name]
    print(f"  {name:8s}: {med:.3f}  -{med-lo:.3f}  +{hi-med:.3f}")


In [ ]:
# Log-evidence: the Bayesian integral over the entire prior volume.
# A higher log Z means a better-fitting model *accounting for complexity*.
# The uncertainty (logz[1]) reflects numerical integration error.
logz, logz_err = results.logz
print(f"Log-evidence: {logz:.3f} ± {logz_err:.3f}")
print()
print("Rule of thumb for model comparison (Jeffreys scale):")
print("  Δ log Z > 1.0  →  substantial evidence for better model")
print("  Δ log Z > 2.5  →  strong evidence")
print("  Δ log Z > 5.0  →  decisive evidence")


In [ ]:
# χ²_red: reduced chi-squared from the weighted posterior average log-likelihood.
# χ²_red ≈ 1 → good fit; > 1 → under-fit; < 1 → over-fit or underestimated errors.
import numpy as np

# Approximate number of data points (replace with actual value if you have the observation)
n_data = 450    # approximate for SINFONI K-band after cropping to 2.0-2.45 µm
n_free = len(results.free_parameters)

# Weighted average log-likelihood over post-burn-in samples
logl_post = results.logl[results.burn_in:]
w_post    = results.weights[results.burn_in:]
best_logL = np.average(logl_post, weights=w_post)

chi2      = -2 * best_logL
chi2_red  = chi2 / (n_data - n_free)

print(f"  n_data   = {n_data}")
print(f"  n_free   = {n_free}")
print(f"  -2 log L = {chi2:.2f}")
print(f"  χ²_red   = {chi2_red:.3f}")


## Section 2: Corner plot customisation

`PLOTS_CONFIG.CornerPlot` is a `CornerPlotConfig` dataclass. Every field maps
directly to a `corner.corner()` keyword argument, plus a few ForMoSA extras.
Use `.set_corner_plot_config(**kwargs)` to change any field before plotting.


In [ ]:
from ForMoSA.core.config import PLOTS_CONFIG
from ForMoSA.nested_sampling.plotting import Plotting
import logging

plotter = Plotting(results, logger=logging.getLogger("tutorial"))

# Defaults — show what the configuration looks like
cfg = PLOTS_CONFIG.CornerPlot
print("Default CornerPlotConfig fields:")
for k, v in cfg.to_dict.items():
    print(f"  {k}: {v}")


In [ ]:
# Customise and plot
PLOTS_CONFIG.CornerPlot.set_corner_plot_config(
    bins=40,                  # histogram bins (default 80 — reduce for speed)
    color="#2E86AB",          # contour + histogram colour
    smooth=0.5,               # Gaussian smoothing of 2D contours (0 = none)
    show_titles=True,         # show "median +X -Y" in axis titles
    quantiles=(0.16, 0.5, 0.84),  # vertical lines in 1D histograms
    plot_datapoints=False,    # don't show individual sample dots
)

fig_corner = plotter.plot_corner()
fig_corner.suptitle("AB Pic b — posterior (customised corner)", y=1.01)
fig_corner.tight_layout()
import matplotlib.pyplot as plt
plt.show()


## Section 3: Chains plot

The chains plot shows the evolution of each sampled parameter over the nested sampling
run. The **burn-in line** marks where the sampler has sufficiently explored the prior;
only samples after this point are used for the posterior. The **weight overlay** shows
how posterior weight is distributed across the chain.


In [ ]:
# Default chains plot
PLOTS_CONFIG.ChainsPlot.set_chains_plot_config(
    color_chains="steelblue",        # chain line colour
    show_weights=True,               # overlay weight trace (right y-axis)
    plot_best_value=True,            # horizontal line at posterior median
    linestyle_burn_in="--",          # burn-in marker style
    color_plot_burn_in="#E84855",    # burn-in marker colour
)

fig_chains, axs = plotter.plot_chains()
fig_chains.suptitle("Sample chains — burn-in (red dashed) and weights (grey)", y=1.01)
plt.show()


## Section 4: Radar plot

The radar plot normalises each parameter to [0, 1] relative to its sample range
and shows the median ± 1σ interval as a filled polygon. It is a compact way to
visualise how tightly each parameter is constrained relative to its prior.


In [ ]:
PLOTS_CONFIG.RadarPlot.set_radar_plot_config(
    quantiles=(0.16, 0.84),      # 1-sigma interval (default)
    color_radar="#2E86AB",
    color_uncertainty="#2E86AB",
    alpha_fill=0.3,
)

fig_radar, ax_radar = plotter.plot_radars()
plt.show()


## Section 5: Best-fit spectrum

`plot_fit` requires the `ObservationSet` and a `best_fit` list (one `ObservedModel`
per observation). These are available on `analysis.ns` after running the fit.
In this tutorial we do not have a live `analysis` object, so we demonstrate
the configuration options only.

`plot_native_model=True` overlays the un-convolved, un-RV-shifted model in
addition to the processed best-fit — useful for checking how much the resolution
and RV shift affect the model.


In [ ]:
# Customise best-fit plot appearance
PLOTS_CONFIG.BestFitPlot.set_best_fit_plot_config(
    color_fit="black",          # best-fit model line colour (default "black")
    color_residuals="#555555",  # residuals colour
    linewidth=1.2,              # model line width
    zorder=200,                 # draw model on top of data
)

print("BestFitPlotConfig set. To actually plot, run from Tutorial 2's notebook:")
print("  analysis.plot(analysis.ns.results, plot_native_model=False)")
print()
print("plot_native_model=True adds the un-convolved, un-shifted model —")
print("useful to verify the adaptation step produced the expected resolution.")


## Section 6: Per-observation colour (MOSAIC context)

In MOSAIC mode you have multiple observations plotted on the same axes. Assign
distinct colours by setting `obs.plot_config.set_plot_config(color=...)` on
each observation before calling `analysis.plot(...)`.


In [ ]:
# Example: colour each observation by its central wavelength using a colormap.
# (This requires a live `analysis` object from Tutorial 4.)
import matplotlib.cm as cm

# Pseudocode — run this from the Tutorial 4 notebook:
# for obs in analysis.ns.restricted_observations:
#     color = cm.inferno(
#         analysis.observations.mcolors_normalize(obs.central_wavelength)
#     )
#     obs.plot_config.set_plot_config(color=color)
# analysis.plot(analysis.ns.results)

print("Run the code above in Tutorial 4's notebook for MOSAIC colour-coding.")


## Section 7: Saving publication-quality figures

`analysis.plot(...)` saves PDFs automatically to `results/`. To customise
the output (format, DPI, bounding box), capture the returned `Figure` object
and call `savefig` directly.


In [ ]:
# analysis.plot() returns a Figure. Capture it:
# fig = analysis.plot(analysis.ns.results)  # or from a Plotting instance:

# From a Plotting instance you can save individual plot components:
fig_corner = plotter.plot_corner()
out_path = Path(".") / "corner_publication.pdf"
fig_corner.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")

# For raster formats (e.g., PNG for slides):
out_png = Path(".") / "corner_publication.png"
fig_corner.savefig(out_png, dpi=200, bbox_inches="tight")
print(f"Saved: {out_png}")


## Section 8: Next steps

- **Tutorial 6 — Cluster / MPI deployment:** Scale up with PyMultiNest and MPI
  for production-quality posteriors (500+ live points, more free parameters).
- **API docs:** `docs/api/` for `Plotting`, `NSResults`, `PlotsConfig`, and all
  config dataclasses with their full parameter lists.
